# CS440 Lab 5: Q-learning starter

Adapted from the group's Lab 3 `markov.ipynb`. This is an unfinished starter,
not a trained robot or a completed Lab 5 submission.

Kept: the environment structure, 80/10/10 movement helpers, start lookup,
and a simplified policy display. The start is now top-left for Lab 4/5.

Removed: value iteration, expected-utility calculations, the explicit
transition table used by the planner, old utilities, the policy runner,
and the three living-cost experiments and their saved outputs.

The class name `MDPEnvironment` is retained from Lab 3; it represents the
simulator, not a value-iteration solver. Q-learning will use experiences
without reading the simulator's transition probabilities or reward map.

## Before running

The original notebook used `%run bootstrap.ipynb` and `aima.agents`.
The bootstrap command was removed because that file was not supplied.
The course's AIMA package is now included in `aima/`.
Install dependencies with `python -m pip install -r requirements.txt` in
the notebook's Python environment. The local package supplies
`aima.agents` without the bootstrap notebook.

TODO functions raise
`NotImplementedError` rather than produce misleading results.


In [2]:
# Uses the local course AIMA package; install requirements.txt first.
from aima.agents import *
import random


## Environment

Movement helpers below are adapted from Lab 3. Reward handling is unfinished.
Finish `execute_action` before using this environment for training.

Agreed rules: 80% intended direction, 10% each perpendicular direction;
blocked moves stay in place. Ordinary and blocked moves receive the living
reward. Entering -1 or +10 ends the episode with exactly that reward.


In [ ]:
default_world = [['start', 0, 0, 0],
                 [0, 'X', 0, 0],
                 [0, 0,   0, 0],
                 [0, -1,  0, 10]]

class MDPEnvironment(Environment):
    def __init__(self, env=default_world, living_reward=-0.04):
            super().__init__()
            self.status = env
            self.living_reward = living_reward

    def percept(self, agent):
            """Returns the agent's location, and the location status."""
            return agent.location, self.status[agent.location[0]][agent.location[1]]

    def execute_action(self, agent, action):
        """TODO: return (next_location, reward, done) for one action."""
        # TODO: reject actions after an episode has ended.
        # TODO: use _attempt with the correct intended/perpendicular moves.
        # TODO: terminal cells return exactly -1 or +10 and done=True.
        # TODO: other moves, including blocked moves, return living_reward.
        # TODO: update agent.performance once using that same reward.
        raise NotImplementedError("TODO: finish Lab 5 rewards and episode handling")

    def _attempt(self, agent, intended_action, accidental_action1, accidental_action2):
        slip = random.random()
        tup = slip < 0.8, slip >= 0.9

        match tup:
            case True, _:
                intended_action(agent)
            case _, False:
                accidental_action1(agent)
            case _, True:
                accidental_action2(agent)

    """ Movements are named so that they correspond to the actual direction.
    For example, an agent at list indices (0, 1)
    could go down to make the new list indices (1, 1),
    or it could go left to make the new list indices (0, 0)"""

    def _down(self, agent):
        newX = agent.location[0] + 1
        Y = agent.location[1]
        if not (newX >= len(self.status) or self.status[newX][Y] == 'X'):
            agent.location = newX, Y

    def _up(self, agent):
        newX = agent.location[0] - 1
        Y = agent.location[1]
        if not (newX < 0 or self.status[newX][Y] == 'X'):
            agent.location = newX, Y

    def _right(self, agent):
        X = agent.location[0]
        newY = agent.location[1] + 1
        if not (newY >= len(self.status[0]) or self.status[X][newY] == 'X'):
            agent.location = X, newY

    def _left(self, agent):
        X = agent.location[0]
        newY = agent.location[1] - 1
        if not (newY < 0 or self.status[X][newY] == 'X'):
            agent.location = X, newY

    def default_location(self, thing):
        for i, list in enumerate(self.status):
            for j, element in enumerate(list):
                if element == 'start':
                    return (i, j)

    def find_recharge_location(self):
        """TODO: find and return the coordinates of the cell labeled 10."""
        # Environment helper requested in the Lab 3 feedback.
        # Keep this separate from default_location, which finds the start.
        raise NotImplementedError("TODO: locate the recharge cell")

    def __str__(self):
        return str(self.status)


In [ ]:
ACTIONS = ['Up', 'Down', 'Left', 'Right']

# Proposed starting values; evaluate them before claiming they work well.
LIVING_REWARD = -0.04
GAMMA = 0.95
ALPHA = 0.1
EPSILON = 0.2
EVALUATION_EPSILON = 0.0


## Q-learning TODOs

1. Add episode reset and finish the environment interface.
2. Initialize Q-values as positions are observed.
3. Select actions with epsilon-greedy exploration, breaking ties randomly.
4. Update Q(s,a) from the observed reward and next state. Do not bootstrap
   from terminal states.
5. Train over repeated episodes with a step limit and reproducible seeds.
6. Evaluate without exploration or Q-value updates, using separate seeds.
7. Print action/path logs, total rewards, and goal/trap/timeout counts.

The learner must not read `env.status`, `_attempt`, or recharge-location
helpers to derive a policy. Only the simulator uses those internals.


In [ ]:
def q_learning(env, episodes, max_steps, alpha=ALPHA, gamma=GAMMA,
               epsilon=EPSILON, seed=0):
    """TODO: learn action values from environment interactions."""
    # TODO: return the learned Q-table and training results.
    raise NotImplementedError("TODO: implement Q-learning")


def evaluate_agent(env, q_values, episodes, max_steps, seed=1):
    """TODO: evaluate the learned policy without changing Q-values."""
    # TODO: use epsilon=0, log paths, and report goal/trap/timeout outcomes.
    raise NotImplementedError("TODO: implement evaluation and path logging")


## Policy display

Call this helper after deriving a policy from the learned Q-table.
The default world is used here only to display obstacles and terminal cells.


In [ ]:
direction = {'Up': 'U', 'Down': 'D', 'Left': 'L', 'Right': 'R'}
def print_policy(policy, world=default_world): 
    print("Policy: ")
    for i in range(len(world)):
        row = []
        for g in range(len(world[0])):
            postion = world[i][g]
            if postion == "X":
                row.append(' X ')
            elif postion == -1:
                row.append('TRAP')
            elif postion == 10:
                row.append('RECHARGE')
            else:
                row.append(f'{direction[policy[(i,g)]]}')
        print(' '.join(row))


## Remaining Lab 5 work

- Install requirements.txt, finish the TODOs, and add meaningful tests.
- Run training and evaluation, then record actual results.
- Add and demonstrate the selected hooks and skills, alongside subagents.
- Update the README with setup and run commands once they work.
- Prepare the private group presentation showing the running solution.
